# Hybrid Pareto Frontier Analysis

This notebook implements a hybrid approach to computing distances to the Pareto frontier:
1. **Compute Pareto frontier using LINEAR cost** (traditional approach)
2. **Calculate distances using LOG cost** (treats cost multiplicatively)

The key challenge: Linear frontier segments become concave curves in log-space, requiring non-linear optimization.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler

import sys
from pathlib import Path
import os

# Add the src directory to Python path
script_dir = Path.cwd()
quantitative_dir = script_dir.parent
src_path = quantitative_dir / 'src'
sys.path.insert(0, str(src_path))

# Change to quantitative directory for proper file paths
os.chdir(quantitative_dir)

# Import dataloader and utilities
from dataloader import load_paper_df, load_most_recent_df
from pareto_utils import compute_pareto_frontier_with_origin, _pareto_indices

In [2]:
# Load data
model_df, agent_df, benchmark_df = load_paper_df()

# Remove Sonnet 4 models (matching existing analysis)
model_df = model_df[~model_df['model'].str.contains('Sonnet 4', na=False)]
benchmark_df = benchmark_df[~benchmark_df['model'].str.contains('Sonnet 4', na=False)]

print(f"Loaded {len(model_df)} model records")
print(f"Loaded {len(benchmark_df)} benchmark records")
print(f"Unique models: {model_df['model'].nunique()}")
print(f"Unique benchmarks: {model_df['benchmark_name'].nunique()}")

Loaded 106 model records
Loaded 150 benchmark records
Unique models: 12
Unique benchmarks: 9


In [3]:
def distance_to_line_segment_logspace(point_log, segment_linear):
    """
    Compute minimum distance from a point in log-cost space to a line segment
    defined in linear-cost space.
    
    The frontier is defined as a straight line in (linear_cost, accuracy) space,
    but we need distance in (log_cost, accuracy) space. This creates a concave
    curve in log-space.
    
    Args:
        point_log: [log10(cost), accuracy] - point in log space
        segment_linear: [[cost1, acc1], [cost2, acc2]] - segment endpoints in linear space
        
    Returns:
        float: Minimum distance from point to the curve
    """
    log_cost_point, acc_point = point_log
    
    # Unpack segment endpoints in linear space
    (cost1, acc1), (cost2, acc2) = segment_linear
    
    # Special case: vertical or horizontal segments
    if abs(cost2 - cost1) < 1e-10:
        # Vertical line at constant cost
        log_cost_line = np.log10(cost1 + 1e-10)
        if acc1 <= acc_point <= acc2 or acc2 <= acc_point <= acc1:
            return abs(log_cost_point - log_cost_line)
        else:
            # Distance to endpoints
            d1 = np.sqrt((log_cost_point - np.log10(cost1 + 1e-10))**2 + (acc_point - acc1)**2)
            d2 = np.sqrt((log_cost_point - np.log10(cost2 + 1e-10))**2 + (acc_point - acc2)**2)
            return min(d1, d2)
    
    # Define the line in linear space: acc = m * cost + b
    m = (acc2 - acc1) / (cost2 - cost1)
    b = acc1 - m * cost1
    
    # Objective: distance squared in log-space from point to curve
    # The curve is parameterized by cost ∈ [cost1, cost2]
    # For each cost, acc = m * cost + b
    # In log space: (log10(cost), m * cost + b)
    def objective(cost):
        if cost <= 0:
            return 1e10  # Penalty for invalid cost
        acc_on_line = m * cost + b
        log_cost_on_line = np.log10(cost)
        return (log_cost_point - log_cost_on_line)**2 + (acc_point - acc_on_line)**2
    
    # Optimize to find closest point on segment
    result = minimize(
        objective,
        x0=[(cost1 + cost2) / 2],  # Start at midpoint
        bounds=[(cost1, cost2)],
        method='L-BFGS-B'
    )
    
    min_dist_sq = result.fun
    
    # Also check endpoints
    log_cost1 = np.log10(cost1 + 1e-10)
    log_cost2 = np.log10(cost2 + 1e-10)
    
    d1_sq = (log_cost_point - log_cost1)**2 + (acc_point - acc1)**2
    d2_sq = (log_cost_point - log_cost2)**2 + (acc_point - acc2)**2
    
    min_dist_sq = min(min_dist_sq, d1_sq, d2_sq)
    
    return np.sqrt(min_dist_sq)

In [4]:
def distance_to_frontier_logspace(point_log, frontier_linear):
    """
    Compute minimum distance from a point in log-cost space to a Pareto frontier
    defined in linear-cost space.
    
    Args:
        point_log: [log10(cost), accuracy] - point in log space
        frontier_linear: Array of shape (n, 2) with frontier points in linear space [(cost, acc), ...]
        
    Returns:
        float: Minimum distance from point to the frontier curve
    """
    if len(frontier_linear) == 0:
        return np.inf
    
    if len(frontier_linear) == 1:
        # Single point frontier
        log_cost_frontier = np.log10(frontier_linear[0][0] + 1e-10)
        return np.sqrt(
            (point_log[0] - log_cost_frontier)**2 + 
            (point_log[1] - frontier_linear[0][1])**2
        )
    
    # Compute distance to each segment
    min_distance = np.inf
    
    for i in range(len(frontier_linear) - 1):
        segment = [frontier_linear[i], frontier_linear[i + 1]]
        dist = distance_to_line_segment_logspace(point_log, segment)
        min_distance = min(min_distance, dist)
    
    return min_distance

In [5]:
def compute_hybrid_frontier_distances(df, cost_col='total_cost', acc_col='accuracy'):
    """
    Compute distances using a hybrid approach:
    1. Compute Pareto frontier using LINEAR cost
    2. Calculate normalized distances using LOG cost
    
    This requires non-linear optimization because the linear frontier becomes
    a concave curve in log-space.
    
    Args:
        df: DataFrame containing benchmark data with columns:
            - benchmark_name: Name of benchmark
            - model: Model name
            - {cost_col}: Cost values
            - {acc_col}: Accuracy values
        cost_col: Name of the cost column (default: 'total_cost')
        acc_col: Name of the accuracy column (default: 'accuracy')
        
    Returns:
        DataFrame with columns:
            - benchmark_name: Name of benchmark
            - model: Model name
            - normalized_distance: Distance to frontier in normalized log-cost space
            - is_on_pareto: Boolean indicating if model is on the Pareto frontier (in linear space)
            - norm_log_cost: Normalized log10(cost) value
            - norm_accuracy: Normalized accuracy value
    """
    required_cols = ["benchmark_name", "model", cost_col, acc_col]
    
    # Validate required columns exist
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Aggregate to model × benchmark means
    plot_df = (
        df.dropna(subset=required_cols)
          .groupby(["benchmark_name", "model"], as_index=False)
          .agg({cost_col: "mean", acc_col: "mean"})
    )
    
    results = []
    
    # Process each benchmark separately
    for bench in sorted(plot_df['benchmark_name'].unique()):
        sub = plot_df[plot_df['benchmark_name'] == bench].copy()
        
        if sub.empty:
            continue
        
        # Extract RAW values
        raw_costs = sub[cost_col].to_numpy()
        raw_accs = sub[acc_col].to_numpy()
        
        # Step 1: Compute Pareto frontier using LINEAR cost
        linear_pareto_frontier = compute_pareto_frontier_with_origin(raw_costs, raw_accs)
        
        # Step 2: Identify which models are on the Pareto frontier (in linear space)
        costs_with_origin = np.concatenate([[0], raw_costs])
        accs_with_origin = np.concatenate([[0], raw_accs])
        p_idx_with_origin = _pareto_indices(costs_with_origin, accs_with_origin)
        
        pareto_model_indices = [idx - 1 for idx in p_idx_with_origin if idx > 0]
        pareto_model_set = set(sub.iloc[pareto_model_indices]['model'].tolist())
        
        # Step 3: Transform to log-space for distance calculation
        epsilon = 1e-10
        log_costs = np.log10(raw_costs + epsilon)
        
        # Step 4: Normalize in log-space
        log_cost_min, log_cost_max = log_costs.min(), log_costs.max()
        acc_min, acc_max = raw_accs.min(), raw_accs.max()
        
        if log_cost_max > log_cost_min:
            norm_log_costs = (log_costs - log_cost_min) / (log_cost_max - log_cost_min)
        else:
            norm_log_costs = np.full(len(log_costs), 0.5)
        
        if acc_max > acc_min:
            norm_accs = (raw_accs - acc_min) / (acc_max - acc_min)
        else:
            norm_accs = np.full(len(raw_accs), 0.5)
        
        # Step 5: Compute distance for each model
        # The frontier is in linear space, but we compute distances in log space
        for idx, (model_name, norm_log_cost, norm_acc) in enumerate(zip(sub['model'], norm_log_costs, norm_accs)):
            # Check if model is on Pareto frontier
            is_on_pareto = model_name in pareto_model_set
            
            if is_on_pareto:
                distance = 0.0
            else:
                # Denormalize point to compute distance
                if log_cost_max > log_cost_min:
                    point_log_cost = norm_log_cost * (log_cost_max - log_cost_min) + log_cost_min
                else:
                    point_log_cost = log_costs[idx]
                
                if acc_max > acc_min:
                    point_acc = norm_acc * (acc_max - acc_min) + acc_min
                else:
                    point_acc = raw_accs[idx]
                
                point_log = [point_log_cost, point_acc]
                
                # Compute distance in denormalized log-space to linear frontier
                distance_denorm = distance_to_frontier_logspace(point_log, linear_pareto_frontier)
                
                # Normalize the distance
                # Scale by the diagonal of the normalized space
                if log_cost_max > log_cost_min and acc_max > acc_min:
                    scale = np.sqrt((log_cost_max - log_cost_min)**2 + (acc_max - acc_min)**2)
                    distance = distance_denorm / scale
                else:
                    distance = distance_denorm
            
            results.append({
                'benchmark_name': bench,
                'model': model_name,
                'normalized_distance': distance,
                'is_on_pareto': is_on_pareto,
                'norm_log_cost': norm_log_cost,
                'norm_accuracy': norm_acc
            })
    
    return pd.DataFrame(results)

In [8]:
# Run the hybrid analysis
print("Computing hybrid frontier distances...")
print("(Frontier computed using LINEAR cost, distances computed using LOG cost)\n")

hybrid_df = compute_hybrid_frontier_distances(benchmark_df)

print(f"Computed distances for {len(hybrid_df)} model-benchmark combinations")
print(f"Models on Pareto frontier: {hybrid_df['is_on_pareto'].sum()} instances")
print(f"\nFirst few rows:")
print(hybrid_df.head(10))

Computing hybrid frontier distances...
(Frontier computed using LINEAR cost, distances computed using LOG cost)

Computed distances for 106 model-benchmark combinations
Models on Pareto frontier: 31 instances

First few rows:
   benchmark_name                   model  normalized_distance  is_on_pareto  \
0  assistantbench         Claude Opus 4.1             0.558266         False   
1  assistantbench    Claude Opus 4.1 High             0.670377         False   
2  assistantbench       Claude-3.7 Sonnet             0.236139         False   
3  assistantbench  Claude-3.7 Sonnet High             0.091761         False   
4  assistantbench             DeepSeek R1             0.315406         False   
5  assistantbench             DeepSeek V3             0.113452         False   
6  assistantbench                 GPT-4.1             0.066820         False   
7  assistantbench            GPT-5 Medium             0.170918         False   
8  assistantbench        Gemini 2.0 Flash             

In [9]:
# Create models × benchmarks pivot table
pivot_table = hybrid_df.pivot_table(
    index='model',
    columns='benchmark_name',
    values='normalized_distance',
    aggfunc='mean'
)

# Add average distance column
pivot_table['Average'] = pivot_table.mean(axis=1)

# Sort by average distance (lower is better)
pivot_table = pivot_table.sort_values('Average')

print("Models × Benchmarks: Normalized Distance to Pareto Frontier")
print("(Frontier: LINEAR cost | Distances: LOG cost)")
print("="*100)
print(pivot_table.round(4))

Models × Benchmarks: Normalized Distance to Pareto Frontier
(Frontier: LINEAR cost | Distances: LOG cost)
benchmark_name          assistantbench  corebench_hard    gaia  \
model                                                            
Gemini 2.0 Flash                0.0154          0.0128  0.0000   
o4-mini Low                     0.0000          0.0165  0.0000   
o4-mini High                    0.0565          0.0000  0.0000   
DeepSeek V3                     0.1135          0.0000  0.0723   
GPT-4.1                         0.0668          0.0361  0.0000   
o3 Medium                       0.0000          0.0662  0.0981   
GPT-5 Medium                    0.1709          0.0128  0.0000   
Claude-3.7 Sonnet High          0.0918          0.0139  0.0748   
Claude-3.7 Sonnet               0.2361          0.0000  0.1363   
DeepSeek R1                     0.3154          0.0541  0.1400   
Claude Opus 4.1                 0.5583          0.0000  0.3381   
Claude Opus 4.1 High            0.67

In [ ]:
# Create styled table with color coding
def style_distance_table(df):
    """
    Apply styling to the pivot table:
    - Green background for 0.0 (on frontier)
    - Gradient for other values (white to red)
    - Bold text for frontier models
    """
    # Create a copy for styling
    styled = df.style
    
    # Color gradient for all columns except 'Average'
    benchmark_cols = [col for col in df.columns if col != 'Average']
    
    styled = styled.background_gradient(
        cmap='RdYlGn_r',  # Red (high) to Green (low)
        subset=benchmark_cols,
        vmin=0,
        vmax=df[benchmark_cols].max().max()
    )
    
    # Highlight the Average column
    styled = styled.background_gradient(
        cmap='RdYlGn_r',
        subset=['Average'],
        vmin=0,
        vmax=df['Average'].max()
    )
    
    # Format to 4 decimal places
    styled = styled.format("{:.4f}")
    
    # Add caption
    styled = styled.set_caption(
        "Hybrid Pareto Analysis: Distance to Frontier (Linear Frontier, Log Distances)<br>"
        "Lower values = closer to frontier = better cost-effectiveness"
    )
    
    return styled

# Display styled table
styled_table = style_distance_table(pivot_table)
styled_table

In [ ]:
# Summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

# Average distance per model (sorted)
model_avg = hybrid_df.groupby('model')['normalized_distance'].mean().sort_values()
print("\nAverage Distance to Frontier by Model (Lower = Better):")
print("-"*50)
for rank, (model, dist) in enumerate(model_avg.items(), 1):
    print(f"{rank:2d}. {model:30s} {dist:.4f}")

# Count of Pareto appearances per model
pareto_counts = hybrid_df[hybrid_df['is_on_pareto']].groupby('model').size().sort_values(ascending=False)
print("\n\nPareto Frontier Appearances by Model:")
print("-"*50)
for model, count in pareto_counts.items():
    total_benchmarks = len(hybrid_df[hybrid_df['model'] == model])
    pct = (count / total_benchmarks) * 100
    print(f"{model:30s} {count:2d}/{total_benchmarks:2d} ({pct:5.1f}%)")

# Average distance per benchmark
benchmark_avg = hybrid_df.groupby('benchmark_name')['normalized_distance'].mean().sort_values()
print("\n\nAverage Distance to Frontier by Benchmark:")
print("-"*50)
for bench, dist in benchmark_avg.items():
    print(f"{bench:30s} {dist:.4f}")